# 🛒 Apriori Algorithm - Market Basket Analysis

The **Apriori Algorithm** is a classic algorithm used in **Association Rule Mining**.
It identifies frequent itemsets in a dataset and derives association rules from them.

### 📌 Key Concepts:

| Term           | Definition                                       |
| -------------- | ------------------------------------------------ |
| **Support**    | How frequently an itemset appears in the dataset |
| **Confidence** | How often the rule is correct                    |
| **Lift**       | Strength of the rule over random co-occurrence   |


## 📦 Step 1: Install & Import Libraries


In [ ]:
# Install required libraries
!sudo pacman -S python-mlxtend python-pandas python-matplotlib python-seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print("✅ All libraries imported successfully!")

[sudo] password for msh: 

## 🛍️ Step 2: Sample Dataset - Grocery Transactions

We simulate a grocery store dataset with **20 customer transactions**.


In [ ]:
# Sample grocery transaction dataset
dataset = [
    ['Milk', 'Bread', 'Butter'],
    ['Beer', 'Bread', 'Diapers'],
    ['Milk', 'Diapers', 'Beer', 'Cola'],
    ['Bread', 'Milk', 'Diapers', 'Beer'],
    ['Bread', 'Milk', 'Butter', 'Cola'],
    ['Beer', 'Cola', 'Diapers'],
    ['Milk', 'Bread', 'Cola'],
    ['Bread', 'Butter', 'Eggs'],
    ['Milk', 'Eggs', 'Cola'],
    ['Bread', 'Milk', 'Eggs', 'Butter'],
    ['Beer', 'Diapers', 'Eggs'],
    ['Milk', 'Bread', 'Diapers'],
    ['Cola', 'Eggs', 'Bread'],
    ['Milk', 'Butter', 'Eggs'],
    ['Beer', 'Cola', 'Bread'],
    ['Milk', 'Bread', 'Beer', 'Diapers'],
    ['Eggs', 'Butter', 'Cola'],
    ['Bread', 'Diapers', 'Cola'],
    ['Milk', 'Beer', 'Eggs'],
    ['Butter', 'Bread', 'Diapers', 'Cola']
]

print(f"📊 Total Transactions: {len(dataset)}")
print("\n🔍 First 5 transactions:")
for i, t in enumerate(dataset[:5]):
    print(f"  Transaction {i+1}: {t}")

## 🔄 Step 3: Data Preprocessing - One-Hot Encoding


In [ ]:
# Encode transactions using TransactionEncoder
te = TransactionEncoder()
te_array = te.fit(dataset).transform(dataset)

# Convert to DataFrame
df = pd.DataFrame(te_array, columns=te.columns_)

print("✅ Encoded Transaction DataFrame:")
print(df.head(10).to_string())
print(f"\n📐 Shape: {df.shape} (Transactions x Items)")

## 📊 Step 4: Item Frequency Analysis


In [ ]:
# Calculate item frequencies
item_freq = df.sum().sort_values(ascending=False)
item_support = (item_freq / len(df)).round(3)

print("📦 Item Frequency & Support:")
freq_df = pd.DataFrame({'Frequency': item_freq, 'Support': item_support})
print(freq_df)

# Plot
plt.figure(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(item_freq)))
bars = plt.bar(item_freq.index, item_freq.values, color=colors, edgecolor='black', linewidth=0.7)
plt.title('🛒 Item Frequency in Transactions', fontsize=15, fontweight='bold')
plt.xlabel('Items', fontsize=12)
plt.ylabel('Frequency Count', fontsize=12)
plt.xticks(fontsize=11)

for bar, val in zip(bars, item_freq.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()
print("\n✅ Bar chart plotted!")

## ⚙️ Step 5: Apply Apriori Algorithm

We find all **frequent itemsets** with minimum support = **0.3** (appear in at least 30% of transactions).


In [ ]:
# Apply Apriori algorithm
frequent_itemsets = apriori(
    df,
    min_support=0.3,
    use_colnames=True
)

# Add itemset length column
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

# Sort by support
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

print(f"✅ Total Frequent Itemsets Found: {len(frequent_itemsets)}")
print("\n📋 All Frequent Itemsets:")
print(frequent_itemsets.to_string(index=True))

## 🔍 Step 6: Filter by Itemset Size


In [ ]:
# 1-itemsets
print("📌 1-Itemsets (Single Items):")
print(frequent_itemsets[frequent_itemsets['length'] == 1].to_string())

print("\n📌 2-Itemsets (Pairs):")
print(frequent_itemsets[frequent_itemsets['length'] == 2].to_string())

print("\n📌 3-Itemsets (Triplets):")
three = frequent_itemsets[frequent_itemsets['length'] == 3]
if len(three) > 0:
    print(three.to_string())
else:
    print("  ❌ No 3-itemsets found at current support threshold.")

## 📐 Step 7: Generate Association Rules

Extract rules with minimum **confidence = 0.6**.


In [ ]:
# Generate association rules
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.6
)

# Select and round relevant columns
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rules[['support', 'confidence', 'lift']] = rules[['support', 'confidence', 'lift']].round(3)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"✅ Total Association Rules Generated: {len(rules)}")
print("\n📋 All Association Rules (sorted by Lift):")
print(rules.to_string())

## 🏆 Step 8: Top Rules by Lift


In [ ]:
# Display top 5 rules
top_rules = rules.head(5)

print("🏆 Top 5 Association Rules by Lift:")
print("-" * 70)
for i, row in top_rules.iterrows():
    ant = ', '.join(list(row['antecedents']))
    con = ', '.join(list(row['consequents']))
    print(f"  Rule {i+1}: {ant} ➡️  {con}")
    print(f"           Support={row['support']}, Confidence={row['confidence']}, Lift={row['lift']}")
    print()

## 📈 Step 9: Visualize - Support vs Confidence (Scatter Plot)


In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    cmap='RdYlGn',
    s=200,
    edgecolors='black',
    linewidths=0.7,
    alpha=0.85
)

cbar = plt.colorbar(scatter)
cbar.set_label('Lift', fontsize=12)

plt.xlabel('Support', fontsize=13)
plt.ylabel('Confidence', fontsize=13)
plt.title('📈 Association Rules: Support vs Confidence\n(Color = Lift)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print("✅ Scatter plot plotted!")

## 🔥 Step 10: Heatmap - Item Co-occurrence Matrix


In [ ]:
# Co-occurrence matrix
co_matrix = df.T.dot(df)
np.fill_diagonal(co_matrix.values, 0)  # Remove self-pairs

plt.figure(figsize=(8, 6))
sns.heatmap(
    co_matrix,
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='gray',
    cbar_kws={'label': 'Co-occurrence Count'}
)
plt.title('🔥 Item Co-occurrence Heatmap', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
print("✅ Heatmap plotted!")

## 🧪 Step 11: Test with Different Support Thresholds


In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5]
itemset_counts = []
rule_counts = []

for thresh in thresholds:
    fi = apriori(df, min_support=thresh, use_colnames=True)
    itemset_counts.append(len(fi))
    if len(fi) > 0:
        r = association_rules(fi, metric='confidence', min_threshold=0.5)
        rule_counts.append(len(r))
    else:
        rule_counts.append(0)

print("📊 Effect of Support Threshold:")
print("-" * 50)
print(f"{'Threshold':<15} {'Freq Itemsets':<20} {'Rules (conf≥0.5)'}")
for t, i, r in zip(thresholds, itemset_counts, rule_counts):
    print(f"{t:<15} {i:<20} {r}")

# Plot
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(thresholds, itemset_counts, 'bo-', linewidth=2, markersize=8, label='Frequent Itemsets')
ax1.set_xlabel('Min Support Threshold', fontsize=12)
ax1.set_ylabel('Frequent Itemsets', color='blue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(thresholds, rule_counts, 'rs--', linewidth=2, markersize=8, label='Association Rules')
ax2.set_ylabel('Association Rules', color='red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='red')

blue_patch = mpatches.Patch(color='blue', label='Frequent Itemsets')
red_patch = mpatches.Patch(color='red', label='Association Rules')
plt.legend(handles=[blue_patch, red_patch], loc='upper right')
plt.title('📉 Impact of Support Threshold on Results', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()
print("✅ Threshold analysis plotted!")

## ✅ Step 12: Summary


In [ ]:
print("=" * 60)
print("          📦 APRIORI ALGORITHM - SUMMARY")
print("=" * 60)
print(f"  Total Transactions        : {len(dataset)}")
print(f"  Unique Items              : {len(te.columns_)}")
print(f"  Min Support Threshold     : 0.30")
print(f"  Min Confidence Threshold  : 0.60")
print(f"  Frequent Itemsets Found   : {len(frequent_itemsets)}")
print(f"  Association Rules Found   : {len(rules)}")
print()
if len(rules) > 0:
    best = rules.iloc[0]
    ant = ', '.join(list(best['antecedents']))
    con = ', '.join(list(best['consequents']))
    print(f"  🏆 Best Rule: {ant} ➡️  {con}")
    print(f"     Lift = {best['lift']}, Confidence = {best['confidence']}")
print("=" * 60)
print("✅ Apriori Algorithm executed successfully!")